<a href="https://colab.research.google.com/github/componavt/topkar-space/blob/main/src/ner/transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Базовый URL для raw-файлов в вашем репозитории
base_url = "https://raw.githubusercontent.com/componavt/topkar-space/main/data/"

# Список файлов для загрузки
files_to_load = ["abbreviations.csv", "test_transformer.csv"]

# Загружаем файлы в словарь pandas DataFrame'ов
dataframes = {}
for filename in files_to_load:
    file_url = base_url + filename
    try:
        df = pd.read_csv(file_url)
        dataframes[filename] = df
        print(f"✅ Успешно загружен {filename}: {len(df)} строк")
    except Exception as e:
        print(f"❌ Ошибка загрузки {filename}: {e}")

# Теперь файлы доступны для работы
abbreviations_df = dataframes.get("abbreviations.csv")
test_df = dataframes.get("test_transformer.csv")

# Пример: посмотреть первые строки
# if abbreviations_df is not None:
    # print("\nПервые 3 строки abbreviations.csv:")
    # print(abbreviations_df.head(3))

if test_df is not None:
    print("\nПервые 3 строки test_transformer.csv:")
    print(test_df.head(3))

In [ ]:
import os
import requests
import pandas as pd

# Создаем папку для данных в Colab
data_dir = '/content'

# Базовый URL для raw-файлов
base_url = "https://raw.githubusercontent.com/componavt/topkar-space/main/data/"

# Список файлов для скачивания
files_to_download = [
    "abbreviations.csv",
    "test_transformer.csv",
]

# Функция для скачивания файла
def download_file(filename, destination_folder):
    file_url = base_url + filename
    local_path = os.path.join(destination_folder, filename)

    try:
        # Скачиваем файл
        response = requests.get(file_url, timeout=10)
        response.raise_for_status()  # Проверяем на ошибки HTTP

        # Сохраняем файл
        with open(local_path, 'wb') as f:
            f.write(response.content)

        # Проверяем размер файла
        file_size = len(response.content)
        print(f"✅ {filename}: {file_size:,} байт -> {local_path}")

        return True
    except requests.exceptions.RequestException as e:
        print(f"❌ {filename}: Ошибка загрузки - {e}")
        return False

# Скачиваем все файлы
print("\n📥 Начинаем скачивание файлов...")
print("-" * 50)

downloaded_files = []
for filename in files_to_download:
    if download_file(filename, data_dir):
        downloaded_files.append(filename)

print("-" * 50)
print(f"\n✅ Скачано {len(downloaded_files)} из {len(files_to_download)} файлов")

# Показываем содержимое папки
print(f"\n📂 Содержимое папки {data_dir}:")
print("-" * 50)
for file in os.listdir(data_dir):
    file_path = os.path.join(data_dir, file)
    file_size = os.path.getsize(file_path)
    print(f"📄 {file} ({file_size:,} байт)")

# Загружаем один из файлов для проверки
print("\n🔍 Проверка: первые 3 строки abbreviations.csv")
test_file = os.path.join(data_dir, 'abbreviations.csv')
if os.path.exists(test_file):
    df_test = pd.read_csv(test_file)
    print(df_test.head(3))

# Сохраняем пути к файлам в переменные для удобства
abbreviations_path = os.path.join(data_dir, 'abbreviations.csv')
test_path = os.path.join(data_dir, 'test_transformer.csv')
train_path = os.path.join(data_dir, 'train_dataset.csv')

print(f"\n📌 Пути к основным файлам:")
print(f"abbreviations.csv: {abbreviations_path}")
print(f"test_transformer.csv: {test_path}")
print(f"train_dataset.csv: {train_path}")

In [ ]:
import pandas as pd

df = pd.read_csv('abbreviations.csv')

# Группировка по расшифровкам с подсчетом уникальных аббревиатур
result = df.groupby('translation').agg({
    'abbreviation': 'count',
    'description': 'count'
}).rename(columns={'abbreviation': 'abbr_count', 'description': 'examples_count'})

# Сортировка по количеству примеров
result = result.sort_values('examples_count', ascending=False)

print(result)
result.to_csv('translation_stats.csv')

In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset
from sklearn.utils import class_weight
import pickle
import os

# Загрузка и подготовка данных
df = pd.read_csv('abbreviations.csv')

# Удаляем классы с менее чем 2 примерами
class_counts = df['translation'].value_counts()
valid_classes = class_counts[class_counts >= 2].index
df = df[df['translation'].isin(valid_classes)]
print(f"Удалено классов с 1 примером: {len(class_counts) - len(valid_classes)}")
print(f"Осталось классов: {len(valid_classes)}")

# БАЛАНСИРОВКА для "г."
print("\n--- Балансировка классов для 'г.' ---")
g_examples = df[df['abbreviation'] == 'г.']
if len(g_examples) > 0:
    mountain_examples = g_examples[g_examples['translation'] == 'гора']
    city_examples = g_examples[g_examples['translation'] == 'город']

    print(f"Примеров 'г. = гора': {len(mountain_examples)}")
    print(f"Примеров 'г. = город': {len(city_examples)}")

    if len(mountain_examples) > 0 and len(city_examples) > 0:
        if len(mountain_examples) < len(city_examples):
            # Аугментация примеров с "горой"
            multiplier = len(city_examples) //len(mountain_examples) + 1
            mountain_examples_augmented = pd.concat([mountain_examples] * multiplier)
            df = pd.concat([df[df['abbreviation'] != 'г.'], city_examples, mountain_examples_augmented])
            print(f"Балансировка выполнена: примеров 'гора' увеличено до {len(mountain_examples_augmented)}")
        elif len(city_examples) < len(mountain_examples):
            # Аугментация примеров с "городом"
            multiplier = len(mountain_examples) // len(city_examples) + 1
            city_examples_augmented = pd.concat([city_examples] * multiplier)
            df = pd.concat([df[df['abbreviation'] != 'г.'], mountain_examples, city_examples_augmented])
            print(f"Балансировка выполнена: примеров 'город' увеличено до {len(city_examples_augmented)}")
        else:
            print("Классы уже сбалансированы")
else:
    print("Нет примеров с аббревиатурой 'г.'")

# Кодирование меток
le = LabelEncoder()
df['label'] = le.fit_transform(df['translation'])

# Токенизация
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class AbbrDataset(Dataset):
    def __init__(self, texts, abbrs, labels):
        contexts = []
        for text, abbr in zip(texts, abbrs):
            if pd.isna(text):
                text = ""
            if abbr == 'г.':
                if 'под' in text.lower() or 'гора' in text.lower() or 'склон' in text.lower() or 'вершин' in text.lower():
                    context = f"ГОРА/ВОЗВЫШЕННОСТЬ: {text} [SEP] {abbr}"
                elif 'в' in text.lower() or 'жив' in text.lower() or 'насел' in text.lower():
                    context = f"НАСЕЛЕННЫЙ_ПУНКТ: {text} [SEP] {abbr}"
                else:
                    context = f"{text} [SEP] {abbr}"
            else:
                context = f"{text} [SEP] {abbr}"
            contexts.append(context)

        self.encodings = tokenizer(contexts, truncation=True, padding=True, max_length=128)
        self.labels = labels

    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()} | {'labels': torch.tensor(self.labels[idx])}

    def __len__(self): return len(self.labels)

# Разделение данных
X = df[['description', 'abbreviation']].values
y = df['label'].values

# Заполняем пропуски
X = np.array([[str(x[0]) if pd.notna(x[0]) else "", str(x[1]) if pd.notna(x[1]) else ""] for x in X])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Вычисление весов классов
unique_classes = np.unique(y_train)
if len(unique_classes) > 1:
    class_weights = class_weight.compute_class_weight('balanced', classes=unique_classes, y=y_train)
    class_weights = torch.tensor(class_weights, dtype=torch.float)
    print(f"\nВеса классов вычислены")
else:
    class_weights = torch.ones(len(unique_classes), dtype=torch.float)

# Создание датасетов
train_dataset = AbbrDataset(X_train[:,0], X_train[:,1], y_train)
test_dataset = AbbrDataset(X_test[:,0], X_test[:,1], y_test)

# Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nИспользуется устройство: {device}")

# Модель
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=len(le.classes_)
)
model.to(device)

# Кастомный Trainer с балансировкой
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if len(unique_classes) > 1:
            loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Параметры обучения
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    eval_steps=100,
    save_steps=200,
    eval_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=3,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

print("\n--- Начало обучения ---")
trainer.train()

# Функция предсказания
def predict(text, abbr):
    if pd.isna(text):
        text = ""

    # Подготовка контекста с балансировкой
    if abbr == 'г.':
        if any(word in text.lower() for word in ['под', 'гора', 'склон', 'вершин', 'восхожд', 'эльбрус', 'эверест']):
            context = f"ГОРА: {text} [SEP] {abbr}"
        elif any(word in text.lower() for word in ['в', 'жив', 'насел', 'москв', 'петерб']):
            context = f"ГОРОД: {text} [SEP] {abbr}"
        else:
            context = f"{text} [SEP] {abbr}"
    else:
        context = f"{text} [SEP] {abbr}"

    # Токенизация
    inputs = tokenizer(context, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Предсказание
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    # Получаем предсказание
    pred = outputs.logits.argmax().item()
    return le.inverse_transform([pred])[0]

# Сохранение
os.makedirs('./abbr_model_improved', exist_ok=True)
model.save_pretrained('./abbr_model_improved')
tokenizer.save_pretrained('./abbr_model_improved')
with open('./abbr_model_improved/label_encoder.pkl', 'wb') as f:
    pickle.dump(le.classes_, f)

print("\n--- Тестирование модели ---")

# Тест 1: Гора
test_text = "Бывшее болото под г. Горелиха."
test_abbr = "г."
print(f"\nТест 1: {test_text}")
print(f"Аббревиатура: {test_abbr}")
print(f"Предсказание: {predict(test_text, test_abbr)} (ожидается: гора)")

# Тест 2: Город
print(f"\nТест 2: г. Москва")
print(f"Аббревиатура: г.")
print(f"Предсказание: {predict('г. Москва', 'г.')} (ожидается: город)")

# Тест 3: Гора
print(f"\nТест 3: под г. Эльбрус")
print(f"Аббревиатура: г.")
print(f"Предсказание: {predict('под г. Эльбрус', 'г.')} (ожидается: гора)")

# Тест 4: Город
print(f"\nТест 4: живу в г. Санкт-Петербург")
print(f"Аббревиатура: г.")
print(f"Предсказание: {predict('живу в г. Санкт-Петербург', 'г.')} (ожидается: город)")

# Тест 5: Гора
print(f"\nТест 5: восхождение на г. Эверест")
print(f"Аббревиатура: г.")
print(f"Предсказание: {predict('восхождение на г. Эверест', 'г.')} (ожидается: гора)")

# Статистика
print("\n--- Статистика после балансировки ---")
print(f"Всего классов: {len(le.classes_)}")
print(f"Всего примеров: {len(df)}")
# print(f"Примеров с 'г. = гора': {len(df[(df['abbreviation'] == 'г.') & (df['translation'] == 'гора)'])}")
# print(f"Примеров с 'г. = город': {len(df[(df['abbreviation'] == 'г.') & (df['translation'] == 'город')])}")

In [ ]:
import torch
import pickle
from transformers import BertTokenizer, BertForSequenceClassification

# Загрузка одним блоком
def load_model(model_path='./abbr_model_improved'):
    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = BertForSequenceClassification.from_pretrained(model_path)
    with open(f'{model_path}/label_encoder.pkl', 'rb') as f:
        classes = pickle.load(f)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    return tokenizer, model, classes, device

# Инициализация при импорте
tokenizer, model, classes, device = load_model()

# Главная функция
def abbr(text, abbr):
    inputs = tokenizer(f"{text} [SEP] {abbr}", return_tensors="pt", max_length=128, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        pred = model(**inputs).logits.argmax().item()

    return classes[pred]

print(abbr("под г. Эльбрус", "г."))  # гора
print(abbr("Река, впадает в Белое море в г. Кемь", "г."))  # город
print(abbr("Мыс р. Кемь", "р."))  # река

In [ ]:
import torch
import pickle
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification
from collections import defaultdict

# ========== ЗАГРУЗКА МОДЕЛИ ==========
def load_model(model_path='./abbr_model_improved'):
    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = BertForSequenceClassification.from_pretrained(model_path)
    with open(f'{model_path}/label_encoder.pkl', 'rb') as f:
        classes = pickle.load(f)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    return tokenizer, model, classes, device

# Инициализация
tokenizer, model, classes, device = load_model()
print(f"Модель загружена на устройство: {device}")
print(f"Всего классов: {len(classes)}")
print(f"Первые 10 классов: {classes[:10]}" if len(classes) > 10 else f"Классы: {classes}")

# ========== ФУНКЦИЯ ПРЕДСКАЗАНИЯ ==========
def predict_abbreviation(text, abbr_symbol):
    """Предсказание расшифровки аббревиатуры"""
    # Преобразуем в строку, если пришло число
    text = str(text) if not pd.isna(text) else ""
    abbr_symbol = str(abbr_symbol) if not pd.isna(abbr_symbol) else ""

    inputs = tokenizer(f"{text} [SEP] {abbr_symbol}", return_tensors="pt", max_length=128, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        pred = model(**inputs).logits.argmax().item()

    return classes[pred]

# ========== ТЕСТИРОВАНИЕ НА ФАЙЛЕ ==========
def test_model_on_file(test_file='test_transformer.csv'):
    """
    Тестирование модели на файле с данными
    Формат файла: description,abbreviation,translation
    """
    print("\n" + "="*70)
    print(f"ТЕСТИРОВАНИЕ МОДЕЛИ НА ФАЙЛЕ: {test_file}")
    print("="*70)

    try:
        # Загрузка тестовых данных
        df_test = pd.read_csv(test_file)
        print(f"Загружено записей для тестирования: {len(df_test)}")
        print(f"Колонки: {list(df_test.columns)}")

        # Показываем первые несколько строк для проверки
        print("\nПервые 3 строки данных:")
        print(df_test.head(3))

    except FileNotFoundError:
        print(f"❌ Файл {test_file} не найден!")
        return [], 0, {}
    except Exception as e:
        print(f"❌ Ошибка при загрузке файла: {e}")
        return [], 0, {}

    # Преобразуем все в строки и обрабатываем пропуски
    df_test['description'] = df_test['description'].fillna('').astype(str)
    df_test['abbreviation'] = df_test['abbreviation'].fillna('').astype(str)
    df_test['translation'] = df_test['translation'].fillna('').astype(str)

    # Словари для хранения результатов
    results = []
    correct_count = 0
    total_count = len(df_test)

    # Детальная статистика по аббревиатурам
    abbr_stats = defaultdict(lambda: {'correct': 0, 'total': 0, 'predictions': [], 'true_values': []})

    print("\n" + "-"*70)
    print("РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ:")
    print("-"*70)

    # Тестирование каждой записи
    for idx, row in df_test.iterrows():
        text = row['description']
        abbreviation = row['abbreviation']
        true_translation = row['translation']

        # Пропускаем пустые строки
        if not text or not abbreviation:
            continue

        # Получаем предсказание
        try:
            predicted = predict_abbreviation(text, abbreviation)
        except Exception as e:
            print(f"❌ Ошибка при предсказании для строки {idx+1}: {e}")
            predicted = "ОШИБКА"

        # Проверяем правильность
        is_correct = (predicted == true_translation)
        if is_correct:
            correct_count += 1

        # Сохраняем результат
        display_text = text[:50] + '...' if len(text) > 50 else text
        results.append({
            'text': display_text,
            'full_text': text,
            'abbreviation': abbreviation,
            'true': true_translation,
            'predicted': predicted,
            'correct': is_correct
        })

        # Обновляем статистику по аббревиатуре
        abbr_stats[abbreviation]['total'] += 1
        abbr_stats[abbreviation]['correct'] += 1 if is_correct else 0
        abbr_stats[abbreviation]['predictions'].append(predicted)
        abbr_stats[abbreviation]['true_values'].append(true_translation)

        # Выводим каждые 5 записей для контроля
        if (idx + 1) % 5 == 0 or idx < 5:
            status = "✓" if is_correct else "✗"
            print(f"{idx+1:3}. [{status}] {abbreviation:3} -> пред: {predicted:15} | верно: {true_translation:15} | {display_text[:40]}")

    # ========== ОБЩАЯ СТАТИСТИКА ==========
    accuracy = correct_count / total_count * 100 if total_count > 0 else 0

    print("\n" + "="*70)
    print("ИТОГОВЫЙ АНАЛИЗ РЕЗУЛЬТАТОВ")
    print("="*70)

    print(f"\n📊 ОБЩАЯ ТОЧНОСТЬ: {accuracy:.2f}% ({correct_count}/{total_count})")

    # Статистика по аббревиатурам
    if abbr_stats:
        print("\n📈 СТАТИСТИКА ПО АББРЕВИАТУРАМ:")
        print("-"*70)
        print(f"{'Аббр.':<8} {'Точность':<12} {'Верно/Всего':<15} {'Распределение ответов':<30}")
        print("-"*70)

        for abbr_symbol, stats in sorted(abbr_stats.items(), key=lambda x: x[1]['total'], reverse=True):
            if stats['total'] > 0:
                abbr_accuracy = stats['correct'] / stats['total'] * 100

                # Анализ распределения предсказаний
                pred_counts = pd.Series(stats['predictions']).value_counts()
                pred_dist = ', '.join([f"{k}({v})" for k, v in pred_counts.head(3).items()])

                print(f"{abbr_symbol:<8} {abbr_accuracy:>6.1f}%      {stats['correct']:3}/{stats['total']:<3}      {pred_dist[:40]}")

    # Детальный анализ ошибок
    errors = [r for r in results if not r['correct']]
    if errors:
        print("\n❌ ТОП-10 ОШИБОК (где модель ошиблась):")
        print("-"*70)

        for i, error in enumerate(errors[:10]):
            print(f"{i+1:2}. {error['abbreviation']:3}: верно '{error['true']:15}' -> предсказано '{error['predicted']:15}' | {error['text'][:40]}")

        # Анализ самых частых ошибок
        print("\n🔄 САМЫЕ ЧАСТЫЕ ОШИБКИ (путаница в классах):")
        print("-"*70)

        confusion = defaultdict(int)
        for error in errors:
            key = f"{error['true']} -> {error['predicted']}"
            confusion[key] += 1

        for (err_type, count) in sorted(confusion.items(), key=lambda x: x[1], reverse=True)[:10]:
            print(f"  {err_type:30}: {count} раз")
    else:
        print("\n✅ ОШИБОК НЕТ! Модель предсказала всё правильно!")

    # Сохранение результатов в CSV
    if results:
        results_df = pd.DataFrame(results)
        results_df.to_csv('test_results_detailed.csv', index=False, encoding='utf-8')
        print(f"\n💾 Детальные результаты сохранены в 'test_results_detailed.csv' ({len(results)} записей)")

    return results, accuracy, abbr_stats

# ========== ЗАПУСК ТЕСТИРОВАНИЯ ==========
if __name__ == "__main__":
    # Тестирование на файле
    results, accuracy, stats = test_model_on_file('test_transformer.csv')

    # Дополнительно: тестирование отдельных примеров
    # print("\n" + "="*70)
    # print("ДОПОЛНИТЕЛЬНЫЕ ТЕСТЫ:")
    # print("="*70)

    # test_cases = [
    #     ("под г. Эльбрус", "г.", "гора"),
    #     ("Река, впадает в Белое море в г. Кемь", "г.", "город"),
    #     ("Мыс р. Кемь", "р.", "река"),
    #     ("Бывшее болото под г. Горелиха", "г.", "гора"),
    #     ("живу в г. Москва", "г.", "город"),
    #     ("Поляна у п. Онежский", "п.", "посёлок"),
    #     ("д. Вама", "д.", "деревня")
    # ]

    # for text, abbr_symbol, expected in test_cases:
    #     predicted = predict_abbreviation(text, abbr_symbol)
    #     status = "✓" if predicted == expected else "✗"
    #     print(f"{status} {abbr_symbol} -> {predicted:15} (ожидалось: {expected:15}) | {text[:40]}")